In [2]:
from pathlib import Path

raw_dir = Path("data/raw/parcels")
raw_dir.mkdir(parents=True, exist_ok=True)

out_file = raw_dir / "travis_parcels.geojson"
print("Saving to:", out_file.resolve())

Saving to: /home/spingil5/data/raw/parcels/travis_parcels.geojson


In [3]:
import requests
import geopandas as gpd
import pandas as pd
from pathlib import Path

base_url = "https://taxmaps.traviscountytx.gov/arcgis/rest/services/Parcels/MapServer/0/query"

all_gdfs = []
offset = 0
page_size = 2000  # service max record count is 2000
batch_num = 1

while True:
    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": page_size
    }

    print(f"Downloading batch {batch_num} with offset={offset}...")
    r = requests.get(base_url, params=params, timeout=120)
    r.raise_for_status()

    data = r.json()

    if "features" not in data or len(data["features"]) == 0:
        print("No more features found. Stopping.")
        break

    gdf = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
    all_gdfs.append(gdf)

    print(f"  Retrieved {len(gdf)} features")
    if len(gdf) < page_size:
        print("Last page reached.")
        break

    offset += page_size
    batch_num += 1

parcels = pd.concat(all_gdfs, ignore_index=True)
parcels = gpd.GeoDataFrame(parcels, geometry="geometry", crs="EPSG:4326")

print("Total parcels downloaded:", len(parcels))
print("Columns:", list(parcels.columns)[:20])

  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 2000 features
  Retrieved 

In [4]:
parcels.head()

,geometry,OBJECTID,PROP_ID,py_owner_name,situs_num,situs_street,situs_zip,situs_address,py_address,abs_subdv_cd,...,deed_num,deed_book_id,deed_book_page,deed_date,legal_desc,timber_use,timber_market,hyperlink,Shape.STArea(),Shape.STLength()
0,"POLYGON ((-97.76231 30.25448, -97.76238 30.254...",369992,100008,DJB INVESTMENT PROPERTY LLC,1201,TX,78704,S 1201 LAMAR BLVD TX 78704,41 DOOLITTLE DR,S13671,...,2014035621TR,None,,1.388470e+12,LOT 1-4 TEMPLER LOTS,None,None,https://stage.travis.prodigycad.com/property-d...,23511.996084,635.161609
1,"POLYGON ((-97.76099 30.25433, -97.76128 30.254...",369993,100012,1219 SOUTH LAMAR VENTURE LLC,1219,TX,78704,S 1219 LAMAR BLVD TX 78704,400 HOWARD ST,S03168,...,2011172140TR,None,,1.321942e+12,LOT 1A COMMERCIAL SQUARE RESUB & LOTS 5-7 OF T...,None,None,https://stage.travis.prodigycad.com/property-d...,102588.491250,1411.564202
2,"POLYGON ((-97.76364 30.25253, -97.76365 30.252...",369994,100015,TEMPLE CENTER SQUARE LTD,1509,TX,78704,S 1509 LAMAR BLVD TX 78704,601 N LAMAR BLVD STE 301,S08658,...,2001096152TR,None,00000,9.923220e+11,LOT 5 LESS 830 SQ FT MAUFRAIS SUBD THE LOT 2&3...,None,None,https://stage.travis.prodigycad.com/property-d...,120189.143567,1785.433882
3,"POLYGON ((-97.76239 30.25208, -97.76225 30.252...",369995,100018,7-ELEVEN INC,1403,TX,78704,S 1403 LAMAR BLVD TX 78704,PO BOX 4900,S08658,...,2014043470TR,None,,1.395724e+12,LOT 2-4 * LESS 3738SF MAUFRAIS SUBD THE,None,None,https://stage.travis.prodigycad.com/property-d...,101159.922594,1310.445623
4,"POLYGON ((-97.76166 30.25489, -97.76162 30.254...",369996,100020,GSGB L P,1109,TX,78704,S 1109 LAMAR BLVD TX 78704,3821 JUNIPER TRACE STE 207,S12658,...,2007035338TR,None,,1.169100e+12,LOT 19-22 BLK 18 SOUTH HEIGHTS,None,None,https://stage.travis.prodigycad.com/property-d...,11117.654694,422.437371


In [5]:
candidate_fields = [
    "parcel_id", "PARCEL_ID", "PROP_ID", "ACCOUNT", "PID",
    "acreage", "ACRES", "Acres",
    "county", "COUNTY",
    "land_use", "LAND_USE", "UseCode", "STATE_CLASS",
    "owner", "OWNER", "OWNER_NAME",
    "geometry"
]

existing = [c for c in candidate_fields if c in parcels.columns]
print("Existing useful fields:", existing)

if "geometry" not in existing:
    existing.append("geometry")

parcels_clean = parcels[existing].copy()
parcels_clean.head()

Existing useful fields: ['PROP_ID', 'geometry']


,PROP_ID,geometry
0,100008,"POLYGON ((-97.76231 30.25448, -97.76238 30.254..."
1,100012,"POLYGON ((-97.76099 30.25433, -97.76128 30.254..."
2,100015,"POLYGON ((-97.76364 30.25253, -97.76365 30.252..."
3,100018,"POLYGON ((-97.76239 30.25208, -97.76225 30.252..."
4,100020,"POLYGON ((-97.76166 30.25489, -97.76162 30.254..."


In [6]:
parcels_clean.to_file(out_file, driver="GeoJSON")
print("Saved:", out_file)

Saved: data/raw/parcels/travis_parcels.geojson


In [7]:
check = gpd.read_file(out_file)
print(check.shape)
check.head()

(373683, 2)


,PROP_ID,geometry
0,100008,"POLYGON ((-97.76231 30.25448, -97.76238 30.254..."
1,100012,"POLYGON ((-97.76099 30.25433, -97.76128 30.254..."
2,100015,"POLYGON ((-97.76364 30.25253, -97.76365 30.252..."
3,100018,"POLYGON ((-97.76239 30.25208, -97.76225 30.252..."
4,100020,"POLYGON ((-97.76166 30.25489, -97.76162 30.254..."


In [10]:
from pathlib import Path
import requests
import geopandas as gpd
import pandas as pd

raw_dir = Path("data/raw/water")
raw_dir.mkdir(parents=True, exist_ok=True)

def download_arcgis_geojson_by_bbox(base_url, bbox, out_file, crs_epsg=4326, page_size=2500):
    """
    bbox = (xmin, ymin, xmax, ymax) in EPSG:4326
    """
    all_gdfs = []
    offset = 0
    batch = 1

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "geometry": f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
            "geometryType": "esriGeometryEnvelope",
            "inSR": 4326,
            "spatialRel": "esriSpatialRelIntersects",
            "resultOffset": offset,
            "resultRecordCount": page_size
        }

        print(f"Downloading batch {batch} from {base_url} ...")
        r = requests.get(base_url, params=params, timeout=180)
        r.raise_for_status()
        data = r.json()

        if "features" not in data or len(data["features"]) == 0:
            print("No more features.")
            break

        gdf = gpd.GeoDataFrame.from_features(data["features"], crs=f"EPSG:{crs_epsg}")
        all_gdfs.append(gdf)

        print(f"  got {len(gdf)} features")

        if len(gdf) < page_size:
            print("Last page reached.")
            break

        offset += page_size
        batch += 1

    if not all_gdfs:
        raise ValueError("No features downloaded for this bbox.")

    out = pd.concat(all_gdfs, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=f"EPSG:{crs_epsg}")
    out.to_file(out_file, driver="GeoJSON")
    print("Saved to:", out_file)
    return out

In [11]:
print("Batches downloaded:", len(all_gdfs))


Batches downloaded: 1


In [12]:
from pathlib import Path
import requests
import geopandas as gpd
import pandas as pd

raw_dir = Path("data/raw/fiber")
raw_dir.mkdir(parents=True, exist_ok=True)

out_file = raw_dir / "fiber_routes.geojson"

# Public ArcGIS FeatureServer layer that supports GeoJSON queries
# If this one fails for you, use the fallback proxy approach below.
base_url = "https://services.arcgis.com/p44fQSGBTnSRR4hh/arcgis/rest/services/Fiber_Optic_view/FeatureServer/0/query"

all_gdfs = []
offset = 0
page_size = 1000
batch = 1

while True:
    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": page_size
    }

    print(f"Downloading fiber batch {batch}...")
    r = requests.get(base_url, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    if "features" not in data or len(data["features"]) == 0:
        print("No more fiber features.")
        break

    gdf = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
    all_gdfs.append(gdf)

    print(f"  got {len(gdf)} features")

    if len(gdf) < page_size:
        print("Last page reached.")
        break

    offset += page_size
    batch += 1

fiber = pd.concat(all_gdfs, ignore_index=True)
fiber = gpd.GeoDataFrame(fiber, geometry="geometry", crs="EPSG:4326")

print("Total fiber features:", len(fiber))
print("Columns:", list(fiber.columns)[:20])

fiber.to_file(out_file, driver="GeoJSON")
print("Saved to:", out_file)

  got 154 features
Last page reached.
Total fiber features: 154
Columns: ['geometry', 'FID', 'Source', 'Owner', 'DelivDate', 'City_Asset', 'Map_Book_P', 'City_Owned', 'County_Own', 'Created_By', 'Creation_D', 'Last_Edito', 'Last_Edit_', 'Shape__Length']
Saved to: data/raw/fiber/fiber_routes.geojson


In [ ]:
print("PARCEL COLUMNS")
print(parcels.columns.tolist())

print("\nFIBER COLUMNS")
print(fiber.columns.tolist())

print("\nFLOWLINE COLUMNS")
print(flow.columns.tolist())

print("\nWATERBODY COLUMNS")
print(waterbodies.columns.tolist())

In [15]:
# Example bbox around Travis County / Austin region
bbox = (-98.25, 30.00, -97.30, 30.75)

flowline_url = "https://hydro.nationalmap.gov/arcgis/rest/services/3DHP_all/FeatureServer/50/query"
waterbody_url = "https://hydro.nationalmap.gov/arcgis/rest/services/3DHP_all/FeatureServer/60/query"

flowlines = download_arcgis_geojson_by_bbox(
    flowline_url,
    bbox,
    "data/raw/water/flowlines.geojson"
)

waterbodies = download_arcgis_geojson_by_bbox(
    waterbody_url,
    bbox,
    "data/raw/water/waterbodies.geojson"
)

print("Flowlines:", flowlines.shape)
print("Waterbodies:", waterbodies.shape)

  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 1284 features
Last page reached.
Saved to: data/raw/water/flowlines.geojson
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 2500 features
  got 1292 features
Last page reached.
Saved to: data/raw/water/waterbodies.geojson
Flowlines: (23784, 36)
Waterbodies: (16292, 13)


In [16]:
import geopandas as gpd

flow = gpd.read_file("data/raw/water/flowlines.geojson")
wb = gpd.read_file("data/raw/water/waterbodies.geojson")

print(flow.shape, wb.shape)
display(flow.head())
display(wb.head())

(23784, 36) (16292, 13)


,OBJECTID,id3dhp,featuredate,mainstemid,gnisid,gnisidlabel,featuretype,featuretypelabel,lengthkm,waterbodyid3dhp,...,pathlength,arbolatesum,divergence,divergencelabel,rtrndivergence,levelpath,terminalpath,workunitid,Shape__Length,geometry
0,759517,JCF7,1694649600000,https://geoconnex.us/usgs/mainstems/4844361,NaN,NaN,1,Channel Line,0.223,NaN,...,None,None,None,None,None,None,None,NHD,259.749764,"LINESTRING (-97.82199 30.62198, -97.8221 30.62..."
1,759524,JCFE,1694649600000,https://geoconnex.us/usgs/mainstems/10757366,NaN,NaN,1,Channel Line,0.387,NaN,...,None,None,None,None,None,None,None,NHD,450.971126,"LINESTRING (-98.1896 30.66177, -98.18946 30.66..."
2,759907,JCPT,1694649600000,https://geoconnex.us/usgs/mainstems/6609813,1373698.0,Fall Creek,5,Waterbody Connector,0.102,LGBZ4,...,None,None,None,None,None,None,None,NHD,118.431570,"LINESTRING (-98.23581 30.44023, -98.23497 30.4..."
3,760204,JCXU,1694649600000,https://geoconnex.us/ref/mainstems/2626420,1367499.0,San Gabriel River,5,Waterbody Connector,0.697,OIZNK,...,None,None,None,None,None,None,None,NHD,811.870704,"LINESTRING (-97.59941 30.65871, -97.59966 30.6..."
4,760260,JCZC,1694649600000,https://geoconnex.us/ref/mainstems/2653616,1335368.0,Elm Creek,5,Waterbody Connector,0.295,KINPC,...,None,None,None,None,None,None,None,NHD,341.271425,"LINESTRING (-97.74098 30.00893, -97.7412 30.00..."


,OBJECTID,id3dhp,featuredate,mainstemid,gnisid,gnisidlabel,featuretype,featuretypelabel,areasqkm,workunitid,Shape__Length,Shape__Area,geometry
0,277923,IFM87,1694649600000,NaN,NaN,NaN,3,Lake,0.003,NHD,272.222469,3983.706560,"POLYGON ((-97.31492 30.41016, -97.3148 30.4101..."
1,277927,IFM8B,1694649600000,NaN,NaN,NaN,3,Lake,0.002,NHD,188.017573,2305.446795,"POLYGON ((-97.31065 30.40366, -97.31072 30.403..."
2,280610,IFOBA,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,123.628837,1062.804489,"POLYGON ((-97.30533 30.42271, -97.30535 30.422..."
3,281153,IFOQP,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,148.054658,1153.422552,"POLYGON ((-97.30461 30.44057, -97.30457 30.440..."
4,282712,IFQ2T,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,147.295475,1272.633844,"POLYGON ((-97.32107 30.40999, -97.32104 30.409..."


In [17]:
import geopandas as gpd

parcels = gpd.read_file("data/raw/parcels/travis_parcels.geojson")
fiber = gpd.read_file("data/raw/fiber/fiber_routes.geojson")
flow = gpd.read_file("data/raw/water/flowlines.geojson")
waterbodies = gpd.read_file("data/raw/water/waterbodies.geojson")

print("Parcels:", parcels.shape)
print("Fiber:", fiber.shape)
print("Flowlines:", flow.shape)
print("Waterbodies:", waterbodies.shape)

Parcels: (373683, 2)
Fiber: (154, 14)
Flowlines: (23784, 36)
Waterbodies: (16292, 13)


In [18]:
print("PARCEL COLUMNS")
print(parcels.columns.tolist())

print("\nFIBER COLUMNS")
print(fiber.columns.tolist())

print("\nFLOWLINE COLUMNS")
print(flow.columns.tolist())

print("\nWATERBODY COLUMNS")
print(waterbodies.columns.tolist())

PARCEL COLUMNS
['PROP_ID', 'geometry']

FIBER COLUMNS
['FID', 'Source', 'Owner', 'DelivDate', 'City_Asset', 'Map_Book_P', 'City_Owned', 'County_Own', 'Created_By', 'Creation_D', 'Last_Edito', 'Last_Edit_', 'Shape__Length', 'geometry']

FLOWLINE COLUMNS
['OBJECTID', 'id3dhp', 'featuredate', 'mainstemid', 'gnisid', 'gnisidlabel', 'featuretype', 'featuretypelabel', 'lengthkm', 'waterbodyid3dhp', 'flowdirection', 'flowdirectionlabel', 'onsurface', 'onsurfacelabel', 'catchmentid3dhp', 'flowpathid3dhp', 'streamlevel', 'startflag', 'terminalflag', 'streamorder', 'streamcalculator', 'hydrosequence', 'dnhydrosequence', 'uphydrosequence', 'dnlevelpath', 'uplevelpath', 'pathlength', 'arbolatesum', 'divergence', 'divergencelabel', 'rtrndivergence', 'levelpath', 'terminalpath', 'workunitid', 'Shape__Length', 'geometry']

WATERBODY COLUMNS
['OBJECTID', 'id3dhp', 'featuredate', 'mainstemid', 'gnisid', 'gnisidlabel', 'featuretype', 'featuretypelabel', 'areasqkm', 'workunitid', 'Shape__Length', 'Shape_

In [19]:
display(parcels.head(5))
display(fiber.head(5))
display(flow.head(5))
display(waterbodies.head(5))

,PROP_ID,geometry
0,100008,"POLYGON ((-97.76231 30.25448, -97.76238 30.254..."
1,100012,"POLYGON ((-97.76099 30.25433, -97.76128 30.254..."
2,100015,"POLYGON ((-97.76364 30.25253, -97.76365 30.252..."
3,100018,"POLYGON ((-97.76239 30.25208, -97.76225 30.252..."
4,100020,"POLYGON ((-97.76166 30.25489, -97.76162 30.254..."


,FID,Source,Owner,DelivDate,City_Asset,Map_Book_P,City_Owned,County_Own,Created_By,Creation_D,Last_Edito,Last_Edit_,Shape__Length,geometry
0,1,Field Verify,stc,1.077149e+12,F8-FBR-001,F8,,,,NaN,MICHAEL.CHAVEZ,1588204800000,596.043160,"LINESTRING (-90.48125 38.78296, -90.48117 38.7..."
1,2,field collected,mix,1.398816e+12,F3-FBR-005,F3,12,12,,NaN,MICHAEL.CHAVEZ,1588204800000,32.637776,"LINESTRING (-90.55594 38.78016, -90.55619 38.7..."
2,3,field collected,mix,1.398816e+12,F3-FBR-006,F3,12,12,,NaN,MICHAEL.CHAVEZ,1588204800000,480.638390,"LINESTRING (-90.55874 38.78266, -90.55816 38.7..."
3,4,Field collected,mix,1.398816e+12,G4-FBR-001,G4,12,12,,NaN,MICHAEL.CHAVEZ,1614211200000,459.319226,"LINESTRING (-90.5423 38.76315, -90.54014 38.76..."
4,5,field collected,mix,1.398816e+12,E2-FBR-001,E2,12,12,,NaN,MICHAEL.CHAVEZ,1588204800000,412.692161,"LINESTRING (-90.56482 38.78588, -90.56467 38.7..."


,OBJECTID,id3dhp,featuredate,mainstemid,gnisid,gnisidlabel,featuretype,featuretypelabel,lengthkm,waterbodyid3dhp,...,pathlength,arbolatesum,divergence,divergencelabel,rtrndivergence,levelpath,terminalpath,workunitid,Shape__Length,geometry
0,759517,JCF7,1694649600000,https://geoconnex.us/usgs/mainstems/4844361,NaN,NaN,1,Channel Line,0.223,NaN,...,None,None,None,None,None,None,None,NHD,259.749764,"LINESTRING (-97.82199 30.62198, -97.8221 30.62..."
1,759524,JCFE,1694649600000,https://geoconnex.us/usgs/mainstems/10757366,NaN,NaN,1,Channel Line,0.387,NaN,...,None,None,None,None,None,None,None,NHD,450.971126,"LINESTRING (-98.1896 30.66177, -98.18946 30.66..."
2,759907,JCPT,1694649600000,https://geoconnex.us/usgs/mainstems/6609813,1373698.0,Fall Creek,5,Waterbody Connector,0.102,LGBZ4,...,None,None,None,None,None,None,None,NHD,118.431570,"LINESTRING (-98.23581 30.44023, -98.23497 30.4..."
3,760204,JCXU,1694649600000,https://geoconnex.us/ref/mainstems/2626420,1367499.0,San Gabriel River,5,Waterbody Connector,0.697,OIZNK,...,None,None,None,None,None,None,None,NHD,811.870704,"LINESTRING (-97.59941 30.65871, -97.59966 30.6..."
4,760260,JCZC,1694649600000,https://geoconnex.us/ref/mainstems/2653616,1335368.0,Elm Creek,5,Waterbody Connector,0.295,KINPC,...,None,None,None,None,None,None,None,NHD,341.271425,"LINESTRING (-97.74098 30.00893, -97.7412 30.00..."


,OBJECTID,id3dhp,featuredate,mainstemid,gnisid,gnisidlabel,featuretype,featuretypelabel,areasqkm,workunitid,Shape__Length,Shape__Area,geometry
0,277923,IFM87,1694649600000,NaN,NaN,NaN,3,Lake,0.003,NHD,272.222469,3983.706560,"POLYGON ((-97.31492 30.41016, -97.3148 30.4101..."
1,277927,IFM8B,1694649600000,NaN,NaN,NaN,3,Lake,0.002,NHD,188.017573,2305.446795,"POLYGON ((-97.31065 30.40366, -97.31072 30.403..."
2,280610,IFOBA,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,123.628837,1062.804489,"POLYGON ((-97.30533 30.42271, -97.30535 30.422..."
3,281153,IFOQP,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,148.054658,1153.422552,"POLYGON ((-97.30461 30.44057, -97.30457 30.440..."
4,282712,IFQ2T,1694649600000,NaN,NaN,NaN,3,Lake,0.001,NHD,147.295475,1272.633844,"POLYGON ((-97.32107 30.40999, -97.32104 30.409..."


In [20]:
print("Parcels CRS:", parcels.crs)
print("Fiber CRS:", fiber.crs)
print("Flowlines CRS:", flow.crs)
print("Waterbodies CRS:", waterbodies.crs)

Parcels CRS: EPSG:4326
Fiber CRS: EPSG:4326
Flowlines CRS: EPSG:4326
Waterbodies CRS: EPSG:4326


In [21]:
parcel_cols = [c.lower() for c in parcels.columns]
print("Possible parcel id fields:", [c for c in parcels.columns if "id" in c.lower() or "parcel" in c.lower()])
print("Possible acreage fields:", [c for c in parcels.columns if "acre" in c.lower() or "area" in c.lower()])
print("Possible land use fields:", [c for c in parcels.columns if "use" in c.lower() or "class" in c.lower() or "zone" in c.lower()])
print("Possible owner fields:", [c for c in parcels.columns if "owner" in c.lower()])

Possible parcel id fields: ['PROP_ID']
Possible acreage fields: []
Possible land use fields: []
Possible owner fields: []


In [22]:
parcels.dtypes

PROP_ID        int32
geometry    geometry
dtype: object

In [23]:
parcels.iloc[:5, :]

,PROP_ID,geometry
0,100008,"POLYGON ((-97.76231 30.25448, -97.76238 30.254..."
1,100012,"POLYGON ((-97.76099 30.25433, -97.76128 30.254..."
2,100015,"POLYGON ((-97.76364 30.25253, -97.76365 30.252..."
3,100018,"POLYGON ((-97.76239 30.25208, -97.76225 30.252..."
4,100020,"POLYGON ((-97.76166 30.25489, -97.76162 30.254..."


In [24]:
parcels.columns.tolist()

['PROP_ID', 'geometry']

In [25]:
fiber.columns.tolist()

['FID',
 'Source',
 'Owner',
 'DelivDate',
 'City_Asset',
 'Map_Book_P',
 'City_Owned',
 'County_Own',
 'Created_By',
 'Creation_D',
 'Last_Edito',
 'Last_Edit_',
 'Shape__Length',
 'geometry']

In [26]:
waterbodies.columns.tolist()

['OBJECTID',
 'id3dhp',
 'featuredate',
 'mainstemid',
 'gnisid',
 'gnisidlabel',
 'featuretype',
 'featuretypelabel',
 'areasqkm',
 'workunitid',
 'Shape__Length',
 'Shape__Area',
 'geometry']

In [27]:
parcels.head(3)

,PROP_ID,geometry
0,100008,"POLYGON ((-97.76231 30.25448, -97.76238 30.254..."
1,100012,"POLYGON ((-97.76099 30.25433, -97.76128 30.254..."
2,100015,"POLYGON ((-97.76364 30.25253, -97.76365 30.252..."


In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# Reload if needed
parcels = gpd.read_file("data/raw/parcels/travis_parcels.geojson")
fiber = gpd.read_file("data/raw/fiber/fiber_routes.geojson")
flow = gpd.read_file("data/raw/water/flowlines.geojson")
waterbodies = gpd.read_file("data/raw/water/waterbodies.geojson")

# Keep only needed columns
parcels = parcels[["PROP_ID", "geometry"]].copy()
fiber = fiber[["geometry"]].copy()
flow = flow[["geometry"]].copy()
waterbodies = waterbodies[["geometry"]].copy()

# Project to a metric CRS for distance/area calculations
target_crs = "EPSG:3083"

parcels_m = parcels.to_crs(target_crs)
fiber_m = fiber.to_crs(target_crs)
flow_m = flow.to_crs(target_crs)
waterbodies_m = waterbodies.to_crs(target_crs)

print(parcels_m.crs, fiber_m.crs, flow_m.crs, waterbodies_m.crs)

EPSG:3083 EPSG:3083 EPSG:3083 EPSG:3083


In [3]:
# Basic parcel geometry features
parcels_m["parcel_id"] = parcels_m["PROP_ID"].astype(str)
parcels_m["parcel_area_sqm"] = parcels_m.geometry.area
parcels_m["parcel_area_sqkm"] = parcels_m["parcel_area_sqm"] / 1_000_000
parcels_m["parcel_area_acres"] = parcels_m["parcel_area_sqm"] / 4046.8564224

centroids = parcels_m.geometry.centroid
centroids_wgs84 = gpd.GeoSeries(centroids, crs=target_crs).to_crs("EPSG:4326")

parcels_m["centroid_lon"] = centroids_wgs84.x
parcels_m["centroid_lat"] = centroids_wgs84.y

parcels_m[["parcel_id", "parcel_area_acres", "centroid_lat", "centroid_lon"]].head()

,parcel_id,parcel_area_acres,centroid_lat,centroid_lon
0,100008,0.539800,30.254526,-97.762072
1,100012,2.355277,30.254116,-97.761695
2,100015,2.759359,30.251923,-97.763529
3,100018,2.322478,30.252483,-97.762859
4,100020,0.255245,30.255026,-97.761774


In [4]:
# Distance from each parcel centroid to nearest fiber route
parcel_centroids_m = parcels_m.copy()
parcel_centroids_m["geometry"] = parcels_m.geometry.centroid

fiber_union = fiber_m.unary_union

parcel_centroids_m["dist_to_fiber_m"] = parcel_centroids_m.geometry.distance(fiber_union)
parcel_centroids_m["dist_to_fiber_km"] = parcel_centroids_m["dist_to_fiber_m"] / 1000.0

parcel_centroids_m[["parcel_id", "dist_to_fiber_km"]].head()

/tmp/ipykernel_830874/2581238994.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  fiber_union = fiber_m.unary_union


,parcel_id,dist_to_fiber_km
0,100008,1150.877151
1,100012,1150.895358
2,100015,1151.195690
3,100018,1151.108373
4,100020,1150.814975


In [9]:
parcels_small = parcels.sample(n=10000, random_state=42).copy()

In [5]:
import geopandas as gpd

# Use a smaller sample first if needed
# parcels_m = parcels_m.sample(n=10000, random_state=42).copy()

parcel_centroids_m = parcels_m.copy()
parcel_centroids_m["geometry"] = parcels_m.geometry.centroid

# keep only geometry from waterbodies
water_nearest = waterbodies_m[["geometry"]].copy()
water_nearest["water_id"] = range(len(water_nearest))

nearest_water = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    water_nearest,
    how="left",
    distance_col="dist_to_waterbody_m"
)

nearest_water["dist_to_waterbody_km"] = nearest_water["dist_to_waterbody_m"] / 1000.0

parcel_centroids_m = parcel_centroids_m.merge(
    nearest_water[["parcel_id", "dist_to_waterbody_m", "dist_to_waterbody_km"]],
    on="parcel_id",
    how="left"
)

parcel_centroids_m[["parcel_id", "dist_to_waterbody_km"]].head()

,parcel_id,dist_to_waterbody_km
0,100008,1.149341
1,100012,1.203940
2,100015,1.244870
3,100018,1.288454
4,100020,1.106145


In [11]:
fiber_nearest = fiber_m[["geometry"]].copy()
fiber_nearest["fiber_id"] = range(len(fiber_nearest))

nearest_fiber = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    fiber_nearest,
    how="left",
    distance_col="dist_to_fiber_m"
)

nearest_fiber["dist_to_fiber_km"] = nearest_fiber["dist_to_fiber_m"] / 1000.0

parcel_centroids_m = parcel_centroids_m.merge(
    nearest_fiber[["parcel_id", "dist_to_fiber_m", "dist_to_fiber_km"]],
    on="parcel_id",
    how="left"
)

parcel_centroids_m[["parcel_id", "dist_to_fiber_km"]].head()

,parcel_id,dist_to_fiber_km
0,100008,1150.877151
1,100012,1150.895358
2,100015,1151.195690
3,100018,1151.108373
4,100020,1150.814975


In [17]:
water_nearest = waterbodies_m[["geometry"]].copy()
water_nearest["water_id"] = range(len(water_nearest))

nearest_water = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    water_nearest,
    how="left",
    distance_col="dist_to_waterbody_m"
)

nearest_water["dist_to_waterbody_km"] = nearest_water["dist_to_waterbody_m"] / 1000.0

nearest_water = nearest_water[["parcel_id", "dist_to_waterbody_m", "dist_to_waterbody_km"]].copy()
nearest_water = nearest_water.drop_duplicates(subset=["parcel_id"])

print(nearest_water.shape)
nearest_water.head()

(372533, 3)


,parcel_id,dist_to_waterbody_m,dist_to_waterbody_km
0,100008,1149.341490,1.149341
1,100012,1203.940014,1.203940
2,100015,1244.870289,1.244870
3,100018,1288.454227,1.288454
4,100020,1106.144984,1.106145


In [18]:
flow_nearest = flow_m[["geometry"]].copy()
flow_nearest["flow_id"] = range(len(flow_nearest))

nearest_flow = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    flow_nearest,
    how="left",
    distance_col="dist_to_flowline_m"
)

nearest_flow["dist_to_flowline_km"] = nearest_flow["dist_to_flowline_m"] / 1000.0

nearest_flow = nearest_flow[["parcel_id", "dist_to_flowline_m", "dist_to_flowline_km"]].copy()
nearest_flow = nearest_flow.drop_duplicates(subset=["parcel_id"])

print(nearest_flow.shape)
nearest_flow.head()

(372533, 3)


,parcel_id,dist_to_flowline_m,dist_to_flowline_km
0,100008,169.822747,0.169823
1,100012,112.650987,0.112651
2,100015,106.385706,0.106386
3,100018,90.419616,0.090420
4,100020,167.543676,0.167544


In [19]:
feature_df = parcels_m[[
    "parcel_id",
    "parcel_area_sqm",
    "parcel_area_sqkm",
    "parcel_area_acres",
    "intersects_waterbody"
]].copy()

feature_df = feature_df.merge(
    parcel_centroids_m[["parcel_id", "centroid_lat", "centroid_lon"]],
    on="parcel_id",
    how="left"
)

feature_df = feature_df.merge(
    nearest_fiber,
    on="parcel_id",
    how="left"
)

feature_df = feature_df.merge(
    nearest_water,
    on="parcel_id",
    how="left"
)

feature_df = feature_df.merge(
    nearest_flow,
    on="parcel_id",
    how="left"
)

def minmax_scale(series):
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

feature_df["acreage_score"] = minmax_scale(feature_df["parcel_area_acres"])
feature_df["fiber_score"] = 1 - minmax_scale(feature_df["dist_to_fiber_km"])
feature_df["water_score"] = 1 - minmax_scale(feature_df["dist_to_waterbody_km"])
feature_df["flowline_score"] = 1 - minmax_scale(feature_df["dist_to_flowline_km"])
feature_df["water_intersection_penalty"] = feature_df["intersects_waterbody"].astype(float)

print("Feature table shape:", feature_df.shape)
print("Unique parcels:", feature_df["parcel_id"].nunique())
feature_df.head()

Feature table shape: (1053404, 18)
Unique parcels: 372533


,parcel_id,parcel_area_sqm,parcel_area_sqkm,parcel_area_acres,intersects_waterbody,centroid_lat,centroid_lon,dist_to_fiber_m,dist_to_fiber_km,dist_to_waterbody_m,dist_to_waterbody_km,dist_to_flowline_m,dist_to_flowline_km,acreage_score,fiber_score,water_score,flowline_score,water_intersection_penalty
0,100008,2184.493802,0.002184,0.539800,0,30.254526,-97.762072,1.150877e+06,1150.877151,1149.341490,1.149341,169.822747,0.169823,0.000132,0.533815,0.938207,0.990448,0.0
1,100012,9531.469622,0.009531,2.355277,0,30.254116,-97.761695,1.150895e+06,1150.895358,1203.940014,1.203940,112.650987,0.112651,0.000577,0.533583,0.935272,0.993664,0.0
2,100015,11166.730957,0.011167,2.759359,0,30.251923,-97.763529,1.151196e+06,1151.195690,1244.870289,1.244870,106.385706,0.106386,0.000676,0.529756,0.933071,0.994016,0.0
3,100018,9398.734932,0.009399,2.322478,0,30.252483,-97.762859,1.151108e+06,1151.108373,1288.454227,1.288454,90.419616,0.090420,0.000569,0.530869,0.930728,0.994914,0.0
4,100020,1032.938809,0.001033,0.255245,0,30.255026,-97.761774,1.150815e+06,1150.814975,1106.144984,1.106145,167.543676,0.167544,0.000062,0.534607,0.940530,0.990576,0.0


In [20]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

feature_df.to_parquet(processed_dir / "features.parquet", index=False)
feature_df.to_csv(processed_dir / "features.csv", index=False)

print("Saved features.parquet and features.csv")

Saved features.parquet and features.csv


In [21]:
feature_df.shape

(1053404, 18)

In [22]:
def dup_count(df, name):
    total = len(df)
    uniq = df["parcel_id"].nunique()
    print(f"{name}: rows={total}, unique_parcels={uniq}, duplicates={total-uniq}")

dup_count(parcels_m[["parcel_id"]], "parcels_m")
dup_count(parcel_centroids_m[["parcel_id"]], "parcel_centroids_m")
dup_count(nearest_fiber[["parcel_id"]], "nearest_fiber")
dup_count(nearest_water[["parcel_id"]], "nearest_water")
dup_count(nearest_flow[["parcel_id"]], "nearest_flow")

parcels_m: rows=373683, unique_parcels=372533, duplicates=1150
parcel_centroids_m: rows=464510, unique_parcels=372533, duplicates=91977
nearest_fiber: rows=372533, unique_parcels=372533, duplicates=0
nearest_water: rows=372533, unique_parcels=372533, duplicates=0
nearest_flow: rows=372533, unique_parcels=372533, duplicates=0


In [23]:
parcels_base = (
    parcels_m[[
        "parcel_id",
        "parcel_area_sqm",
        "parcel_area_sqkm",
        "parcel_area_acres",
        "intersects_waterbody"
    ]]
    .drop_duplicates(subset=["parcel_id"])
    .copy()
)

centroids_base = (
    parcel_centroids_m[[
        "parcel_id",
        "centroid_lat",
        "centroid_lon"
    ]]
    .drop_duplicates(subset=["parcel_id"])
    .copy()
)

fiber_base = (
    nearest_fiber[[
        "parcel_id",
        "dist_to_fiber_m",
        "dist_to_fiber_km"
    ]]
    .drop_duplicates(subset=["parcel_id"])
    .copy()
)

water_base = (
    nearest_water[[
        "parcel_id",
        "dist_to_waterbody_m",
        "dist_to_waterbody_km"
    ]]
    .drop_duplicates(subset=["parcel_id"])
    .copy()
)

flow_base = (
    nearest_flow[[
        "parcel_id",
        "dist_to_flowline_m",
        "dist_to_flowline_km"
    ]]
    .drop_duplicates(subset=["parcel_id"])
    .copy()
)

In [24]:
feature_df = parcels_base.merge(
    centroids_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)

feature_df = feature_df.merge(
    fiber_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)

feature_df = feature_df.merge(
    water_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)

feature_df = feature_df.merge(
    flow_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)

print("Feature rows:", len(feature_df))
print("Unique parcel_id:", feature_df["parcel_id"].nunique())

Feature rows: 372533
Unique parcel_id: 372533


In [25]:
import numpy as np
import pandas as pd

def minmax_scale(series):
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

feature_df["acreage_score"] = minmax_scale(feature_df["parcel_area_acres"])
feature_df["fiber_score"] = 1 - minmax_scale(feature_df["dist_to_fiber_km"])
feature_df["water_score"] = 1 - minmax_scale(feature_df["dist_to_waterbody_km"])
feature_df["flowline_score"] = 1 - minmax_scale(feature_df["dist_to_flowline_km"])
feature_df["water_intersection_penalty"] = feature_df["intersects_waterbody"].astype(float)

print(feature_df.shape)
display(feature_df.head())

(372533, 18)


,parcel_id,parcel_area_sqm,parcel_area_sqkm,parcel_area_acres,intersects_waterbody,centroid_lat,centroid_lon,dist_to_fiber_m,dist_to_fiber_km,dist_to_waterbody_m,dist_to_waterbody_km,dist_to_flowline_m,dist_to_flowline_km,acreage_score,fiber_score,water_score,flowline_score,water_intersection_penalty
0,100008,2184.493802,0.002184,0.539800,0,30.254526,-97.762072,1.150877e+06,1150.877151,1149.341490,1.149341,169.822747,0.169823,0.000132,0.533815,0.938207,0.990448,0.0
1,100012,9531.469622,0.009531,2.355277,0,30.254116,-97.761695,1.150895e+06,1150.895358,1203.940014,1.203940,112.650987,0.112651,0.000577,0.533583,0.935272,0.993664,0.0
2,100015,11166.730957,0.011167,2.759359,0,30.251923,-97.763529,1.151196e+06,1151.195690,1244.870289,1.244870,106.385706,0.106386,0.000676,0.529756,0.933071,0.994016,0.0
3,100018,9398.734932,0.009399,2.322478,0,30.252483,-97.762859,1.151108e+06,1151.108373,1288.454227,1.288454,90.419616,0.090420,0.000569,0.530869,0.930728,0.994914,0.0
4,100020,1032.938809,0.001033,0.255245,0,30.255026,-97.761774,1.150815e+06,1150.814975,1106.144984,1.106145,167.543676,0.167544,0.000062,0.534607,0.940530,0.990576,0.0


In [26]:
print("Rows:", len(feature_df))
print("Unique parcels:", feature_df["parcel_id"].nunique())

print("\nNull counts:")
print(feature_df.isnull().sum())

print("\nScore summary:")
print(feature_df[[
    "acreage_score",
    "fiber_score",
    "water_score",
    "flowline_score",
    "water_intersection_penalty"
]].describe())

Rows: 372533
Unique parcels: 372533

Null counts:
parcel_id                     0
parcel_area_sqm               0
parcel_area_sqkm              0
parcel_area_acres             0
intersects_waterbody          0
centroid_lat                  0
centroid_lon                  0
dist_to_fiber_m               0
dist_to_fiber_km              0
dist_to_waterbody_m           0
dist_to_waterbody_km          0
dist_to_flowline_m            0
dist_to_flowline_km           0
acreage_score                 0
fiber_score                   0
water_score                   0
flowline_score                0
water_intersection_penalty    0
dtype: int64

Score summary:
       acreage_score    fiber_score    water_score  flowline_score  \
count  372533.000000  372533.000000  372533.000000   372533.000000   
mean        0.000382       0.614611       0.969385        0.983970   
std         0.004839       0.157530       0.021535        0.011877   
min         0.000000       0.000000       0.000000        0.00000

In [27]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

feature_df.to_parquet(processed_dir / "features.parquet", index=False)
feature_df.to_csv(processed_dir / "features.csv", index=False)

print("Saved clean features.")

Saved clean features.


In [28]:
print(fiber_m.total_bounds)
print(parcels_m.total_bounds)

[2323890.90558277 8324516.1014728  2331935.69795125 8335465.80578273]
[1674095.10664874 7303523.68542633 1752201.6114958  7392958.03516628]


In [29]:
import numpy as np
import pandas as pd

def minmax_scale(series):
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

feature_df["acreage_score"] = minmax_scale(feature_df["parcel_area_acres"])
feature_df["water_score"] = 1 - minmax_scale(feature_df["dist_to_waterbody_km"])
feature_df["flowline_score"] = 1 - minmax_scale(feature_df["dist_to_flowline_km"])
feature_df["water_intersection_penalty"] = feature_df["intersects_waterbody"].astype(float)

score_cols = [
    "acreage_score",
    "water_score",
    "flowline_score",
    "water_intersection_penalty"
]

feature_df[score_cols].describe()

,acreage_score,water_score,flowline_score,water_intersection_penalty
count,372533.000000,372533.000000,372533.000000,372533.000000
mean,0.000382,0.969385,0.983970,0.025308
std,0.004839,0.021535,0.011877,0.157059
min,0.000000,0.000000,0.000000,0.000000
25%,0.000038,0.957549,0.978613,0.000000
50%,0.000051,0.974301,0.986628,0.000000
75%,0.000084,0.985744,0.992429,0.000000
max,1.000000,1.000000,1.000000,1.000000


In [30]:
feature_df["baseline_score"] = (
    0.40 * feature_df["acreage_score"] +
    0.30 * feature_df["water_score"] +
    0.20 * feature_df["flowline_score"] -
    0.10 * feature_df["water_intersection_penalty"]
)

top20 = feature_df.sort_values("baseline_score", ascending=False).head(20).copy()

print(top20[[
    "parcel_id",
    "parcel_area_acres",
    "dist_to_waterbody_km",
    "dist_to_flowline_km",
    "intersects_waterbody",
    "baseline_score"
]])

       parcel_id  parcel_area_acres  dist_to_waterbody_km  \
349087    936012        4084.438429              1.672013   
185856    354620        2849.044222              1.708720   
185202    353731        3160.410252              1.173744   
7124      109645        2580.040741              0.304806   
361622    963956        1565.332287              1.425914   
348636    934035        1362.761849              1.467550   
186406    355294        1373.693797              1.358185   
185201    353730        1258.261626              1.147585   
312490    840515        2223.802417              1.085758   
17482     123209        1041.238555              1.041589   
65843     190607        1812.499341              0.285351   
219110    464293         944.141230              1.484664   
57551     178567        1737.072167              0.372842   
82015     214211        1724.976261              0.133805   
332682    893807         738.547967              0.825502   
184792    353217        

In [31]:
top20.describe(include="all")

,parcel_id,parcel_area_sqm,parcel_area_sqkm,parcel_area_acres,intersects_waterbody,centroid_lat,centroid_lon,dist_to_fiber_m,dist_to_fiber_km,dist_to_waterbody_m,dist_to_waterbody_km,dist_to_flowline_m,dist_to_flowline_km,acreage_score,fiber_score,water_score,flowline_score,water_intersection_penalty,baseline_score
count,20,2.000000e+01,20.000000,20.000000,20.000000,20.000000,20.000000,2.000000e+01,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
unique,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,936012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,6.962961e+06,6.962961,1720.585204,0.450000,30.394531,-97.908608,1.145885e+06,1145.885257,1081.467235,1.081467,245.514566,0.245515,0.421254,0.597423,0.941857,0.986190,0.450000,0.603297
std,NaN,3.576656e+06,3.576656,883.810986,0.510418,0.128538,0.176544,9.725392e+03,9.725392,593.365422,0.593365,180.960859,0.180961,0.216385,0.123924,0.031901,0.010179,0.510418,0.065117
min,NaN,2.696709e+06,2.696709,666.371224,0.000000,30.194146,-98.092345,1.134732e+06,1134.732435,133.805121,0.133805,1.348798,0.001349,0.163149,0.394123,0.881220,0.963589,0.000000,0.551888
25%,NaN,4.117061e+06,4.117061,1017.347874,0.000000,30.299933,-98.027711,1.137984e+06,1137.983949,626.876167,0.626876,122.991302,0.122991,0.249079,0.506827,0.920869,0.979424,0.000000,0.556294
50%,NaN,6.605240e+06,6.605240,1632.190406,0.000000,30.350629,-97.985801,1.142878e+06,1142.878283,1116.671655,1.116672,195.705159,0.195705,0.399612,0.635739,0.939964,0.988992,0.000000,0.576985
75%,NaN,7.751046e+06,7.751046,1915.325110,1.000000,30.538082,-97.856545,1.152995e+06,1152.995096,1471.828194,1.471828,365.812842,0.365813,0.468932,0.698104,0.966297,0.993082,1.000000,0.614565


In [33]:
print(feature_df.isnull().sum().sort_values(ascending=False))

parcel_id                     0
parcel_area_sqm               0
parcel_area_sqkm              0
parcel_area_acres             0
intersects_waterbody          0
centroid_lat                  0
centroid_lon                  0
dist_to_fiber_m               0
dist_to_fiber_km              0
dist_to_waterbody_m           0
dist_to_waterbody_km          0
dist_to_flowline_m            0
dist_to_flowline_km           0
acreage_score                 0
fiber_score                   0
water_score                   0
flowline_score                0
water_intersection_penalty    0
baseline_score                0
dtype: int64


In [34]:
feature_df["baseline_score_v2"] = (
    0.45 * feature_df["acreage_score"] +
    0.30 * feature_df["water_score"] +
    0.15 * feature_df["flowline_score"] -
    0.25 * feature_df["water_intersection_penalty"]
)

top20_v2 = feature_df.sort_values("baseline_score_v2", ascending=False).head(20).copy()

print(top20_v2[[
    "parcel_id",
    "parcel_area_acres",
    "dist_to_waterbody_km",
    "dist_to_flowline_km",
    "intersects_waterbody",
    "baseline_score_v2"
]])

       parcel_id  parcel_area_acres  dist_to_waterbody_km  \
185856    354620        2849.044222              1.708720   
349087    936012        4084.438429              1.672013   
361622    963956        1565.332287              1.425914   
348636    934035        1362.761849              1.467550   
186406    355294        1373.693797              1.358185   
185201    353730        1258.261626              1.147585   
17482     123209        1041.238555              1.041589   
185202    353731        3160.410252              1.173744   
219110    464293         944.141230              1.484664   
184792    353217         896.970405              1.894698   
332682    893807         738.547967              0.825502   
38418     153135         945.675833              2.209299   
259831    564023         666.371224              0.711554   
261735    566701         597.149393              0.705943   
185857    354621         848.208589              2.715598   
349085    936010        

In [39]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

parcels = gpd.read_file("data/raw/parcels/travis_parcels.geojson")
fiber = gpd.read_file("data/raw/fiber/fiber_routes.geojson")
flow = gpd.read_file("data/raw/water/flowlines.geojson")
waterbodies = gpd.read_file("data/raw/water/waterbodies.geojson")

parcels = parcels[["PROP_ID", "geometry"]].copy()
fiber = fiber[["geometry"]].copy()
flow = flow[["geometry"]].copy()
waterbodies = waterbodies[["geometry"]].copy()

target_crs = "EPSG:3083"

parcels_m = parcels.to_crs(target_crs)
fiber_m = fiber.to_crs(target_crs)
flow_m = flow.to_crs(target_crs)
waterbodies_m = waterbodies.to_crs(target_crs)

parcels_m["parcel_id"] = parcels_m["PROP_ID"].astype(str)
parcels_m["parcel_area_sqm"] = parcels_m.geometry.area
parcels_m["parcel_area_sqkm"] = parcels_m["parcel_area_sqm"] / 1_000_000
parcels_m["parcel_area_acres"] = parcels_m["parcel_area_sqm"] / 4046.8564224

parcel_centroids_m = parcels_m[["parcel_id", "geometry"]].copy()
parcel_centroids_m["geometry"] = parcel_centroids_m.geometry.centroid

centroids_wgs84 = parcel_centroids_m.to_crs("EPSG:4326")
parcel_centroids_m["centroid_lon"] = centroids_wgs84.geometry.x.values
parcel_centroids_m["centroid_lat"] = centroids_wgs84.geometry.y.values

fiber_nearest = fiber_m[["geometry"]].copy()
fiber_nearest["fiber_id"] = range(len(fiber_nearest))

nearest_fiber = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    fiber_nearest,
    how="left",
    distance_col="dist_to_fiber_m"
)
nearest_fiber["dist_to_fiber_km"] = nearest_fiber["dist_to_fiber_m"] / 1000.0
nearest_fiber = nearest_fiber[["parcel_id", "dist_to_fiber_m", "dist_to_fiber_km"]].drop_duplicates(subset=["parcel_id"])

water_nearest = waterbodies_m[["geometry"]].copy()
water_nearest["water_id"] = range(len(water_nearest))

nearest_water = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    water_nearest,
    how="left",
    distance_col="dist_to_waterbody_m"
)
nearest_water["dist_to_waterbody_km"] = nearest_water["dist_to_waterbody_m"] / 1000.0
nearest_water = nearest_water[["parcel_id", "dist_to_waterbody_m", "dist_to_waterbody_km"]].drop_duplicates(subset=["parcel_id"])

flow_nearest = flow_m[["geometry"]].copy()
flow_nearest["flow_id"] = range(len(flow_nearest))

nearest_flow = gpd.sjoin_nearest(
    parcel_centroids_m[["parcel_id", "geometry"]],
    flow_nearest,
    how="left",
    distance_col="dist_to_flowline_m"
)
nearest_flow["dist_to_flowline_km"] = nearest_flow["dist_to_flowline_m"] / 1000.0
nearest_flow = nearest_flow[["parcel_id", "dist_to_flowline_m", "dist_to_flowline_km"]].drop_duplicates(subset=["parcel_id"])

water_intersections = gpd.sjoin(
    parcels_m[["parcel_id", "geometry"]],
    waterbodies_m[["geometry"]],
    how="left",
    predicate="intersects"
)
water_intersections["intersects_waterbody"] = water_intersections["index_right"].notna().astype(int)
water_intersections_flag = (
    water_intersections.groupby("parcel_id", as_index=False)["intersects_waterbody"].max()
)

parcels_m = parcels_m.merge(
    water_intersections_flag,
    on="parcel_id",
    how="left"
)
parcels_m["intersects_waterbody"] = parcels_m["intersects_waterbody"].fillna(0).astype(int)

parcels_base = parcels_m[[
    "parcel_id",
    "parcel_area_sqm",
    "parcel_area_sqkm",
    "parcel_area_acres",
    "intersects_waterbody"
]].drop_duplicates(subset=["parcel_id"]).copy()

centroids_base = parcel_centroids_m[[
    "parcel_id",
    "centroid_lat",
    "centroid_lon"
]].drop_duplicates(subset=["parcel_id"]).copy()

fiber_base = nearest_fiber.drop_duplicates(subset=["parcel_id"]).copy()
water_base = nearest_water.drop_duplicates(subset=["parcel_id"]).copy()
flow_base = nearest_flow.drop_duplicates(subset=["parcel_id"]).copy()

feature_df = parcels_base.merge(
    centroids_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)
feature_df = feature_df.merge(
    fiber_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)
feature_df = feature_df.merge(
    water_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)
feature_df = feature_df.merge(
    flow_base,
    on="parcel_id",
    how="left",
    validate="one_to_one"
)

def minmax_scale(series):
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

feature_df["acreage_score"] = minmax_scale(feature_df["parcel_area_acres"])
feature_df["water_score"] = 1 - minmax_scale(feature_df["dist_to_waterbody_km"])
feature_df["flowline_score"] = 1 - minmax_scale(feature_df["dist_to_flowline_km"])
feature_df["water_intersection_penalty"] = feature_df["intersects_waterbody"].astype(float)

feature_df["baseline_score_v2"] = (
    0.45 * feature_df["acreage_score"] +
    0.30 * feature_df["water_score"] +
    0.15 * feature_df["flowline_score"] -
    0.25 * feature_df["water_intersection_penalty"]
)

top20_v2 = feature_df.sort_values("baseline_score_v2", ascending=False).head(20).copy()

required_cols = [
    "parcel_id",
    "parcel_area_acres",
    "intersects_waterbody",
    "centroid_lat",
    "centroid_lon",
    "dist_to_waterbody_km",
    "dist_to_flowline_km",
    "acreage_score",
    "water_score",
    "flowline_score",
    "water_intersection_penalty",
    "baseline_score_v2"
]

feature_df_clean = feature_df[required_cols].dropna().copy()

Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

feature_df_clean.to_parquet("data/processed/features_clean.parquet", index=False)
feature_df_clean.to_csv("data/processed/features_clean.csv", index=False)
top20_v2.to_csv("outputs/top20_baseline_v2.csv", index=False)

feature_df_clean.head(), top20_v2.head()

(  parcel_id  parcel_area_acres  intersects_waterbody  centroid_lat  \
 0    100008           0.539800                     0     30.254526   
 1    100012           2.355277                     0     30.254116   
 2    100015           2.759359                     0     30.251923   
 3    100018           2.322478                     0     30.252483   
 4    100020           0.255245                     0     30.255026   
 
    centroid_lon  dist_to_waterbody_km  dist_to_flowline_km  acreage_score  \
 0    -97.762072              1.149341             0.169823       0.000132   
 1    -97.761695              1.203940             0.112651       0.000577   
 2    -97.763529              1.244870             0.106386       0.000676   
 3    -97.762859              1.288454             0.090420       0.000569   
 4    -97.761774              1.106145             0.167544       0.000062   
 
    water_score  flowline_score  water_intersection_penalty  baseline_score_v2  
 0     0.938207      